In [35]:
# Deep Learning
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Utilities
import numpy as np
import pandas as pd
import os

print(" TensorFlow Version:", tf.__version__)

 TensorFlow Version: 2.21.0


In [36]:
DATA_PATH = os.path.join("..", "dataset", "training")

X_train = np.load(os.path.join(DATA_PATH, "X_train.npy"))
y_train = np.load(os.path.join(DATA_PATH, "y_train.npy"))

X_val = np.load(os.path.join(DATA_PATH, "X_val.npy"))
y_val = np.load(os.path.join(DATA_PATH, "y_val.npy"))

X_test = np.load(os.path.join(DATA_PATH, "X_test.npy"))
y_test = np.load(os.path.join(DATA_PATH, "y_test.npy"))

print("✅ Training Set   :", X_train.shape, y_train.shape)
print("✅ Validation Set :", X_val.shape, y_val.shape)
print("✅ Test Set       :", X_test.shape, y_test.shape)

✅ Training Set   : (7608, 154, 63) (7608,)
✅ Validation Set : (275, 154, 63) (275,)
✅ Test Set       : (487, 154, 63) (487,)


In [37]:

print("========== DATASET CHECK ==========")
print("Train shape :", X_train.shape, y_train.shape)
print("Val shape   :", X_val.shape, y_val.shape)
print("Test shape  :", X_test.shape, y_test.shape)

print("\nTrain labels :", len(np.unique(y_train)))
print("Val labels   :", len(np.unique(y_val)))
print("Test labels  :", len(np.unique(y_test)))

print("\nLabel range (train):", y_train.min(), "to", y_train.max())
print("Label range (val):  ", y_val.min(), "to", y_val.max())
print("Label range (test): ", y_test.min(), "to", y_test.max())

========== DATASET CHECK ==========
Train shape : (7608, 154, 63) (7608,)
Val shape   : (275, 154, 63) (275,)
Test shape  : (487, 154, 63) (487,)

Train labels : 210
Val labels   : 210
Test labels  : 210

Label range (train): 0 to 209
Label range (val):   0 to 209
Label range (test):  0 to 209


In [38]:
num_classes = len(np.unique(y_train))

print("Number of classes :", num_classes)
print("Label range        :", y_train.min(), "to", y_train.max())

print("\nTrain labels:", len(np.unique(y_train)))
print("Validation labels:", len(np.unique(y_val)))
print("Test labels:", len(np.unique(y_test)))

print("\nInput shape per video:", X_train.shape[1:])

Number of classes : 210
Label range        : 0 to 209

Train labels: 210
Validation labels: 210
Test labels: 210

Input shape per video: (154, 63)


In [39]:
print("X_train shape:", X_train.shape)
print("Min value:", np.min(X_train))
print("Max value:", np.max(X_train))
print("Mean value:", np.mean(X_train))
print("Std value:", np.std(X_train))

print("\nFirst frame, first 10 values:")
print(X_train[0, 0, :10])

X_train shape: (7608, 154, 63)
Min value: -0.11235343
Max value: 1.0514196
Mean value: 0.12131035
Std value: 0.25054622

First frame, first 10 values:
[ 0.6201981   0.83584845 -0.00150778  0.61943     0.83668894 -0.01466912
  0.60627645  0.86827654 -0.01421977  0.6148244 ]


In [40]:
# ==============================
# Normalize MediaPipe Landmarks
# ==============================

mean = X_train.mean(axis=(0, 1), keepdims=True)
std = X_train.std(axis=(0, 1), keepdims=True)

# Avoid division by zero
std[std == 0] = 1.0

X_train = (X_train - mean) / std
X_val   = (X_val - mean) / std
X_test  = (X_test - mean) / std

print("✅ Normalization complete!")
print("Train mean:", np.mean(X_train))
print("Train std :", np.std(X_train))
print("Val mean  :", np.mean(X_val))
print("Test mean :", np.mean(X_test))

✅ Normalization complete!
Train mean: -2.0026541e-06
Train std : 0.999643
Val mean  : -0.0031216166
Test mean : 0.0021326372


In [41]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Masking,
    LSTM,
    Bidirectional,
    Dense,
    Dropout,
    BatchNormalization
)

model = Sequential([
    # Ignore padded frames
    Masking(mask_value=0.0, input_shape=(154, 63)),

    # First BiLSTM
    Bidirectional(LSTM(128, return_sequences=True)),
    BatchNormalization(),
    Dropout(0.3),

    # Second BiLSTM
    Bidirectional(LSTM(128)),
    BatchNormalization(),
    Dropout(0.3),

    # Dense layer
    Dense(128, activation="relu"),
    Dropout(0.4),

    # Output layer
    Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

c:\Users\palak\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\masking.py:48: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ masking_2 (Masking)             │ (None, 154, 63)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_6 (Bidirectional) │ (None, 154, 256)       │       196,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 154, 256)       │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 154, 256)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_7 (Bidirectional) │ (None, 256)            │       394,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 210)            │        27,090 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 652,882 (2.49 MB)

 Trainable params: 651,858 (2.49 MB)

 Non-trainable params: 1,024 (4.00 KB)

In [42]:
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

early_stop = EarlyStopping(
    monitor="val_accuracy",
    patience=8,
    restore_best_weights=True,
    verbose=1
)

checkpoint = ModelCheckpoint(
    "../models/best_lstm_model.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-5,
    verbose=1
)

In [43]:
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)
import os

# ==============================
# Create models folder
# ==============================
os.makedirs("../models", exist_ok=True)

# ==============================
# Early Stopping
# Stop if validation accuracy doesn't improve
# ==============================
early_stop = EarlyStopping(
    monitor="val_accuracy",
    patience=8,
    restore_best_weights=True,
    verbose=1
)

# ==============================
# Save best model
# ==============================
checkpoint = ModelCheckpoint(
    "../models/best_lstm_model.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

# ==============================
# Reduce Learning Rate if stuck
# ==============================
reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-5,
    verbose=1
)

# ==============================
# Train Model
# ==============================
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=16,
    callbacks=[early_stop, reduce_lr, checkpoint],
    verbose=1
)

Epoch 1/50
476/476 ━━━━━━━━━━━━━━━━━━━━ 0s 755ms/step - accuracy: 0.0088 - loss: 5.5183
Epoch 1: val_accuracy improved from None to 0.00727, saving model to ../models/best_lstm_model.keras

Epoch 1: finished saving model to ../models/best_lstm_model.keras
476/476 ━━━━━━━━━━━━━━━━━━━━ 436s 774ms/step - accuracy: 0.0088 - loss: 5.5183 - val_accuracy: 0.0073 - val_loss: 5.1839 - learning_rate: 0.0010
Epoch 2/50
476/476 ━━━━━━━━━━━━━━━━━━━━ 0s 484ms/step - accuracy: 0.0116 - loss: 5.0904
Epoch 2: val_accuracy improved from 0.00727 to 0.01455, saving model to ../models/best_lstm_model.keras

Epoch 2: finished saving model to ../models/best_lstm_model.keras
476/476 ━━━━━━━━━━━━━━━━━━━━ 233s 490ms/step - accuracy: 0.0116 - loss: 5.0904 - val_accuracy: 0.0145 - val_loss: 4.9423 - learning_rate: 0.0010
Epoch 3/50
476/476 ━━━━━━━━━━━━━━━━━━━━ 0s 476ms/step - accuracy: 0.0221 - loss: 4.8421
Epoch 3: val_accuracy improved from 0.01455 to 0.02909, saving model to ../models/best_lstm_model.keras

Ep

In [44]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=1)

print(f"Test Accuracy : {test_acc*100:.2f}%")
print(f"Test Loss     : {test_loss:.4f}")

16/16 ━━━━━━━━━━━━━━━━━━━━ 8s 368ms/step - accuracy: 0.6345 - loss: 1.7044
Test Accuracy : 63.45%
Test Loss     : 1.7044
